# Finalize: all data in one collection (2004-2026.09)

**notebook version**: `4` (29 Oct 2026)

## ℹ️ About this notebook

This notebook collects all data available so far into one file, on one 30-minute timestamp from
the start of the meteo record in 2004 to 30 September 2026. The datasets of a release are cut from
it by period, for example FP2026.1 (2016 to September 2026) or a single setup period such as
`2026_1`:

```python
finalize.loc['2026-01-01 00:15':'2026-09-30 23:45']
```

It collects three kinds of data:

| Source | What it adds | Period |
|---|---|---|
| `L1 fluxes`: `20_L1_FLUXES/IRGA75/99_IRGA75_L1_MERGED_noSHC_2004-2017_2019` and `20_L1_FLUXES/IRGA72/99_IRGA72_L1_MERGED_2016-2026.09` | Level-1 fluxes: the EddyPro output of the LI-7500 (`IRGA75`) until 2015 and of the LI-7200 (`IRGA72`) from 2016 | 30 March 2004 to September 2026 |
| `opco IRGA75`: `30_FLUX_POST-PROCESSING_CHAIN/IRGA75/01_FPPC_IRGA75_INPUT_OPCO_2004-2017_2019` | the CO2 flux of the LI-7500 corrected with opco (`FC_L1.1_OPCO`) and the columns of the correction (`_OPCO`) | 2004 to 2015 |
| `meteo`: `10_METEO/30_PRODUCTS/99_METEO_MERGED_2004-2026.09` | all meteo: every product that `99` merges | 2004 to September 2026 (`08` to `13` to December 2025) |
| `FPPC`: `30_FLUX_POST-PROCESSING_CHAIN/99_FPPC_MERGED_2004-2026.09` | the flux post-processing of `NEE` (with its partitioning), `LE` (with `ET`) and `H`, both analysers | 2004 to September 2026 |

The fluxes come from one gas analyser per period, as in the flux post-processing chain: the
open-path LI-7500 until 31 December 2015, the enclosed-path LI-7200 from 1 January 2016.
`FLAG_FLUX_SOURCE` gives the analyser of each half-hour (75 or 72), for the Level-1 columns too.
The two analysers share most Level-1 column names; columns that only one of them has are empty in
the other period, and columns without any value in the collection are left out. The LI-7500 run of
2016-2017 and its stand-in run of 2019 are not used.

The meteo comes from `99` only: `99` collects the meteo products, and a product reaches this file by
being merged there. `99` merges all thirteen products. `01` to `07` reach September 2026; `08` to
`13` (`PREC`, `SWC`, `TS`, `G`, `SW_OUT`, `LW_OUT`) still end in December 2025 and are empty from
January 2026 until they are extended.

Each source adds only the columns the file does not have yet. A column that two sources share must
hold the same values in both. The Level-1 files carry `SW_IN_POT` from EddyPro, which runs about 15 minutes
ahead of the timestamp, as if it were computed for the end of each half-hour. The flux notebooks
compute their own `SW_IN_POT` for the middle of each half-hour, from which their day and night
flags come. The collection keeps that one as `SW_IN_POT` and the EddyPro one as
`SW_IN_POT_EDDYPRO`.

The last section lists every variable of the collection with its unit, a short description, its
source and its period. The units and descriptions come from `variable_info.py` in this folder.

## ⚙️ Settings

In [1]:
DATADIR = (r"F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data"
           r"\workflow")
FILES = {
    'L1 fluxes IRGA75': rf"{DATADIR}\20_L1_FLUXES\IRGA75\99_IRGA75_L1_MERGED_noSHC_2004-2017_2019.parquet",
    'L1 fluxes IRGA72': rf"{DATADIR}\20_L1_FLUXES\IRGA72\99_IRGA72_L1_MERGED_2016-2026.09.parquet",
    'opco IRGA75': rf"{DATADIR}\30_FLUX_POST-PROCESSING_CHAIN\IRGA75\01_FPPC_IRGA75_INPUT_OPCO_2004-2017_2019.parquet",
    'meteo': rf"{DATADIR}\10_METEO\30_PRODUCTS\99_METEO_MERGED_2004-2026.09.parquet",
    'FPPC': rf"{DATADIR}\30_FLUX_POST-PROCESSING_CHAIN\99_FPPC_MERGED_2004-2026.09.parquet",
}
PERIOD_START, PERIOD_END = '2004-01-01 00:15', '2026-09-30 23:45'
# The gas analyser of each period, as in the flux post-processing chain (stage 99).
ANALYSER_PERIODS = {'IRGA75': ('2004-01-01 00:15', '2015-12-31 23:45'),
                    'IRGA72': ('2016-01-01 00:15', '2026-09-30 23:45')}

OUTNAME = "10_FLUXES_and_METEO_2004-2026.09"  # same name as this notebook
OUTPATH = rf"{DATADIR}\80_FINALIZE"

In [2]:
from datetime import datetime
import importlib.metadata

import diive as dv
import numpy as np
import pandas as pd

from variable_info import describe_all, info_section

NOTEBOOK_START = datetime.now()
print(f"Last run: {NOTEBOOK_START.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"diive version: v{importlib.metadata.version('diive')}")

Last run: 2026-10-31 16:48:24
diive version: v0.93.0


## 📂 Load data

In [3]:
frames = {k: dv.load_parquet(filepath=v) for k, v in FILES.items()}
for name, frame in frames.items():
    assert frame.index.is_unique, f"{name} has duplicate timestamps"
    print(f"{name:>17}: {frame.shape[0]:,} records x {frame.shape[1]:>3} columns, "
          f"{frame.index[0]} -> {frame.index[-1]}")

> Loaded .parquet file 
F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\20_L1_FLUXES\IRGA
75\99_IRGA75_L1_MERGED_noSHC_2004-2017_2019.parquet (1.810 seconds).

> Loaded .parquet file 
F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\20_L1_FLUXES\IRGA
72\99_IRGA72_L1_MERGED_2016-2026.09.parquet (1.875 seconds).

> Loaded .parquet file 
F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\30_FLUX_POST-PROC
ESSING_CHAIN\IRGA75\01_FPPC_IRGA75_INPUT_OPCO_2004-2017_2019.parquet (1.568 seconds).

> Loaded .parquet file 
F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\10_METEO\30_PRODU
CTS\99_METEO_MERGED_2004-2026.09.parquet (0.293 seconds).

> Loaded .parquet file 
F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\30_FLUX_POST-PROC
ESSING_CHAIN\99_FPPC_MERGED_2004-2026.09.parquet (0.168 seconds).

 L1 fluxes IRGA75: 262,473 records x 320 columns, 2004-03-30 14:45:00 -> 2019-03-20 18:45:00
 L1 fluxes IRGA72: 187,937 records x 329 columns, 2016-01-11 15:45:00 -> 2026-09-30 23:45:00
      opco IRGA75: 280,512 records x 349 columns, 2004-01-01 00:15:00 -> 2019-12-31 23:45:00
            meteo: 398,832 records x  61 columns, 2004-01-01 00:15:00 -> 2026-09-30 23:45:00
             FPPC: 398,832 records x 205 columns, 2004-01-01 00:15:00 -> 2026-09-30 23:45:00


## Collect

In [4]:
def add_new_columns(main, frame, name, source):
    """Add the columns of frame that main does not have yet, on main's timestamps.

    A column that main already has must hold the same values in frame, so that no result is
    dropped silently. Returns the extended frame and records each new column's source.
    """
    assert frame.index.isin(main.index).all(), f"{name} has timestamps outside the main index"
    frame = frame.reindex(main.index)
    new = [c for c in frame.columns if c not in main.columns]
    for c in frame.columns.difference(new):
        a, b = main[c], frame[c]
        both = a.notna() & b.notna()
        assert np.allclose(a[both], b[both], equal_nan=True), f"{c} differs between main and {name}"
    for c in new:
        source[c] = name
    print(f"{name}: {len(new)} new columns, {frame.shape[1] - len(new)} already present")
    return pd.concat([main, frame[new]], axis=1)

In [5]:
# The sources of the collection. The Level-1 fluxes of the two analysers become one source, each
# analyser in its period, with EddyPro's potential radiation renamed so the one of the flux chain
# takes its name. The opco columns cover the LI-7500 period only.
_l1 = []
for _a, (_s, _e) in ANALYSER_PERIODS.items():
    _l1.append(frames[f'L1 fluxes {_a}'].loc[_s:_e])
_l1 = pd.concat(_l1).rename(columns={'SW_IN_POT': 'SW_IN_POT_EDDYPRO'})
assert _l1.index.is_unique
_l1 = _l1.loc[:, _l1.notna().any()]  # columns without any value in the periods used
_opco = frames['opco IRGA75'].loc[slice(*ANALYSER_PERIODS['IRGA75'])]
_opco = _opco[[c for c in _opco.columns if c.endswith('_OPCO')]]
SOURCES = {'L1 fluxes': _l1, 'opco IRGA75': _opco, 'meteo': frames['meteo'], 'FPPC': frames['FPPC']}
SOURCE_FILES = {'L1 fluxes': [FILES['L1 fluxes IRGA75'], FILES['L1 fluxes IRGA72']],
                'opco IRGA75': [FILES['opco IRGA75']], 'meteo': [FILES['meteo']], 'FPPC': [FILES['FPPC']]}

index = pd.date_range(PERIOD_START, PERIOD_END, freq='30min', name='TIMESTAMP_MIDDLE')
finalize = pd.DataFrame(index=index)
SOURCE = {}
for name, frame in SOURCES.items():
    finalize = add_new_columns(finalize, frame, name, SOURCE)
assert finalize.columns.is_unique

# The Level-1 values come from the analyser that FLAG_FLUX_SOURCE names.
for _a, _code in (('IRGA75', 75), ('IRGA72', 72)):
    _s, _e = ANALYSER_PERIODS[_a]
    assert (finalize.loc[_s:_e, 'FLAG_FLUX_SOURCE'] == _code).all(), f"FLAG_FLUX_SOURCE is not {_code} in {_s} - {_e}"

fullset = finalize  # name used by the export cell
DESCRIBED = describe_all(finalize.columns)  # unit and description of every column
print(f"Finalize: {finalize.shape[0]:,} records x {finalize.shape[1]} columns, "
      f"{finalize.memory_usage().sum() / 1e9:.1f} GB in memory")

L1 fluxes: 329 new columns, 0 already present
opco IRGA75: 10 new columns, 0 already present
meteo: 61 new columns, 0 already present
FPPC: 205 new columns, 0 already present
Finalize: 398,832 records x 605 columns, 1.9 GB in memory


### 📋 Table: columns per source

In [6]:
display(pd.Series(SOURCE).value_counts().reindex(list(SOURCES)).rename('columns').to_frame())

,columns
L1 fluxes,329
opco IRGA75,10
meteo,61
FPPC,205


### 📋 Table: coverage per source and year

The share of records per year in which a source has at least one value. A source that ends before
30 September 2026 shows 0 in the years after its end.

In [7]:
_cov = {}
for _name in SOURCES:
    _cols = [c for c, s in SOURCE.items() if s == _name]
    _cov[_name] = finalize[_cols].notna().any(axis=1).groupby(finalize.index.year).mean() * 100
with pd.option_context('display.max_rows', None):
    display(pd.DataFrame(_cov).round(0).astype(int).rename_axis('year'))

,L1 fluxes,opco IRGA75,meteo,FPPC
year,,,,
2004,74,100,100,100
2005,98,100,100,100
2006,98,100,100,100
2007,98,100,100,100
2008,98,100,100,100
2009,97,100,100,100
2010,95,100,100,100
2011,98,100,100,100
2012,92,100,100,100


## 💾 Export

The collection as parquet, and next to it an `_INFO.md` file that lists every column with its unit and a short description, per source.

In [8]:
filepath = dv.save_parquet(data=finalize, filename=OUTNAME, outpath=OUTPATH)

_lines = [f"# {OUTNAME}", "",
          f"{fullset.index[0]} to {fullset.index[-1]} ({fullset.index.name}, UTC+01:00), "
          f"{fullset.shape[0]:,} records x {fullset.shape[1]} columns.",
          f"Written {datetime.now():%Y-%m-%d %H:%M} by `{OUTNAME}.ipynb`.", "",
          "| source | file |", "|---|---|"]
_lines += [f"| {k} | {', '.join(f'`{f}`' for f in v)} |" for k, v in SOURCE_FILES.items()]
_lines += ["", "Fluxes: LI-7500 (IRGA75) " + " - ".join(ANALYSER_PERIODS['IRGA75']) + ", LI-7200 (IRGA72) "
           + " - ".join(ANALYSER_PERIODS['IRGA72']) + "; `FLAG_FLUX_SOURCE` gives the analyser of each half-hour."]
for _name in SOURCES:
    _lines += info_section(_name, [c for c, s in SOURCE.items() if s == _name], DESCRIBED)
with open(rf"{OUTPATH}\{OUTNAME}_INFO.md", "w", encoding="utf-8") as _f:
    _f.write("\n".join(_lines) + "\n")
print(f"Wrote {OUTPATH}\\{OUTNAME}_INFO.md")

> Saved file 
F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\80_FINALIZE\10_FL
UXES_and_METEO_2004-2026.09.parquet (5.956 seconds).

Wrote F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\80_FINALIZE\10_FLUXES_and_METEO_2004-2026.09_INFO.md


## 📋 Variables in the collection

Every variable of the collection, in the order of the file: its unit, a short description, its
source, the number of records with a value, and the first and last of these records.

### 📋 Table: all variables

In [9]:
_valid = finalize.notna()
VARIABLES = pd.DataFrame({
    'unit': pd.Series({c: u for c, (u, d) in DESCRIBED.items()}),
    'description': pd.Series({c: d for c, (u, d) in DESCRIBED.items()}),
    'source': pd.Series(SOURCE),
    'records': _valid.sum(),
    'first': finalize.apply(lambda s: s.first_valid_index()),
    'last': finalize.apply(lambda s: s.last_valid_index()),
}).reindex(finalize.columns)
VARIABLES.index.name = 'variable'
VARIABLES.insert(0, '#', range(1, len(VARIABLES) + 1))
print(f"{len(VARIABLES)} variables; without any value: {int((VARIABLES['records'] == 0).sum())}")
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(VARIABLES)

605 variables; without any value: 0


,#,unit,description,source,records,first,last
variable,,,,,,,
AIR_CP,1,J kg-1 K-1,Specific heat of moist air at constant pressure,L1 fluxes,378918,2004-03-30 14:45:00,2026-09-30 23:45:00
AIR_DENSITY,2,kg m-3,Density of moist air,L1 fluxes,378918,2004-03-30 14:45:00,2026-09-30 23:45:00
AIR_MV,3,m3 mol-1,Molar volume of moist air,L1 fluxes,378918,2004-03-30 14:45:00,2026-09-30 23:45:00
AIR_RHO_CP,4,J m-3 K-1,Volumetric heat capacity of moist air,L1 fluxes,378918,2004-03-30 14:45:00,2026-09-30 23:45:00
AOA_METHOD,5,-,Angle-of-attack correction (EddyPro code),L1 fluxes,378918,2004-03-30 14:45:00,2026-09-30 23:45:00
AXES_ROTATION_METHOD,6,-,Coordinate rotation method (EddyPro code),L1 fluxes,378918,2004-03-30 14:45:00,2026-09-30 23:45:00
BADM_HEIGHTC,7,m,Canopy height,L1 fluxes,378918,2004-03-30 14:45:00,2026-09-30 23:45:00
BADM_INSTPAIR_EASTWARD_SEP_GA_CO2,8,cm,"Eastward separation of the analyser from the sonic, CO2",L1 fluxes,360420,2004-03-31 11:15:00,2026-09-30 23:45:00
BADM_INSTPAIR_EASTWARD_SEP_GA_H2O,9,cm,"Eastward separation of the analyser from the sonic, H2O",L1 fluxes,346461,2004-03-31 11:15:00,2026-09-30 23:45:00


***

# End of notebook.

In [10]:
NOTEBOOK_END = datetime.now()
_h, _rem = divmod(int((NOTEBOOK_END - NOTEBOOK_START).total_seconds()), 3600)
_m, _s = divmod(_rem, 60)

print(f"Started:  {NOTEBOOK_START.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Finished: {NOTEBOOK_END.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Runtime:  {_h:02d}:{_m:02d}:{_s:02d} (hh:mm:ss)")

Started:  2026-10-31 16:48:24
Finished: 2026-10-31 16:48:38
Runtime:  00:00:14 (hh:mm:ss)
